# Part 1: Fully connected networks (poker hands)

Data preparation, experiments (network size, activation function, dropout), cross-validation, and export + reload of the best network.

## 1. Data preparation

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

SEED = 42
DATA_PATH = Path("../data/poker_hands.csv")
TEST_FILE = Path("poker_test_hands.csv")      # raw test hands, used by the reload cell at the end
CLASSES = ["nothing", "one_pair", "two_pairs", "three_of_a_kind", "straight",
           "flush", "full_house", "four_of_a_kind", "straight_flush", "royal_flush"]
SUITS = ["hearts", "diamonds", "clubs", "spades"]
RANK_ENCODING = "onehot"                      # "onehot", or "scaled" = rank as one number in 0..1

In [ ]:
def encode(frame, rank_encoding=RANK_ENCODING):
    """Raw rows (suit1, rank1, ..., suit5, rank5) -> feature matrix. Also used for the exported model."""
    def one_hot(cols, categories):
        return np.hstack([pd.get_dummies(pd.Categorical(frame[c], categories=categories)).to_numpy(np.float32)
                          for c in cols])
    suits = one_hot([f"suit{i}" for i in range(1, 6)], SUITS)            # suits are categories, not numbers
    rank_cols = [f"rank{i}" for i in range(1, 6)]
    if rank_encoding == "onehot":
        ranks = one_hot(rank_cols, list(range(1, 14)))                    # lets the network "see" equal ranks
    else:
        ranks = frame[rank_cols].to_numpy(np.float32) / 13.0              # ranks as one number scaled to 0..1
    return np.hstack([suits, ranks])

def labels(frame):
    return frame["ranking"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()   # integer labels 0..9

df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates().reset_index(drop=True)      # identical hands could otherwise end up in train AND test
X, y = encode(df), labels(df)
print(X.shape, "features per hand")

(1022771, 85) features per hand


In [ ]:
idx_tr, idx_tmp = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEED)
X_tr, y_tr, X_val, y_val, X_test, y_test = X[idx_tr], y[idx_tr], X[idx_val], y[idx_val], X[idx_test], y[idx_test]
df.iloc[idx_test].to_csv(TEST_FILE, index=False)      # the test hands in raw form, for the reload demo

def balance(X, y, cap=20_000, floor=2_000, seed=SEED):
    """Undersample classes above `cap`; oversample (with replacement) classes below `floor`."""
    rng = np.random.default_rng(seed)
    idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        if len(ci) > cap:
            ci = rng.choice(ci, cap, replace=False)
        elif len(ci) < floor:
            ci = rng.choice(ci, floor, replace=True)
        idx.append(ci)
    idx = rng.permutation(np.concatenate(idx))
    return X[idx], y[idx]

X_train, y_train = balance(X_tr, y_tr)               # only the training set is balanced
Y_train, Y_val, Y_test = (to_categorical(a, 10) for a in (y_train, y_val, y_test))
counts = lambda a: np.bincount(a, minlength=10)
print(pd.DataFrame({"class": CLASSES, "train (before)": counts(y_tr), "train (balanced)": counts(y_train),
                    "validation": counts(y_val), "test": counts(y_test)}).to_string(index=False))

          class  train (before)  train (balanced)  validation  test
        nothing          358803             20000       76887 76886
       one_pair          302511             20000       64824 64824
      two_pairs           34102             20000        7308  7307
three_of_a_kind           15109             15109        3238  3238
       straight            2778              2778         595   596
          flush            1432              2000         307   307
     full_house            1021              2000         218   219
 four_of_a_kind             165              2000          35    36
 straight_flush              12              2000           3     2
    royal_flush               6              2000           1     1


## 2. Experiments

In [ ]:
MAX_EPOCHS, PATIENCE, BATCH = 40, 5, 256
RESULTS, MODELS = Path("results"), Path("models")
RESULTS.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)

def build_model(hidden, activation="relu", dropout=0.0):
    """hidden = list of layer sizes, e.g. [128, 128]; dropout is applied after every hidden layer."""
    stack = [keras.Input((X_train.shape[1],))]
    for units in hidden:
        stack.append(layers.Dense(units, activation=activation))
        if dropout:
            stack.append(layers.Dropout(dropout))
    stack.append(layers.Dense(10, activation="softmax"))        # one probability per hand type
    model = keras.Sequential(stack)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
def report(r):
    print(f"{r['name']:24s} epochs={r['epochs_run']:2d}  {r['seconds']:5.0f}s  val_acc={r['val_acc']:.4f}  test_acc={r['test_acc']:.4f}")

def run_experiment(name, hidden, activation="relu", dropout=0.0, force=False):
    path = RESULTS / f"{name}.json"
    if path.exists() and not force:                              # finished runs are loaded, not repeated
        result = json.loads(path.read_text())
        report(result)
        return result
    keras.utils.set_random_seed(SEED)
    model = build_model(hidden, activation, dropout)
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    t = time.perf_counter()
    h = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=MAX_EPOCHS,
                  batch_size=BATCH, callbacks=[stop], verbose=0)
    seconds = time.perf_counter() - t
    test_loss, test_acc = model.evaluate(X_test, Y_test, batch_size=4096, verbose=0)
    pred = model.predict(X_test, batch_size=4096, verbose=0).argmax(1)
    recall = [float((pred[y_test == c] == c).mean()) for c in range(10)]    # per-class accuracy on the test set
    hist = {k: [float(v) for v in vals] for k, vals in h.history.items()}
    best = int(np.argmin(hist["val_loss"]))
    result = dict(name=name, hidden=hidden, activation=activation, dropout=dropout,
                  epochs_run=len(hist["loss"]), best_epoch=best + 1, seconds=round(seconds, 1),
                  val_acc=hist["val_accuracy"][best], val_loss=hist["val_loss"][best],
                  test_acc=float(test_acc), test_loss=float(test_loss),
                  recall=recall, mean_recall=float(np.mean(recall)), history=hist)
    model.save(MODELS / f"{name}.keras")
    path.write_text(json.dumps(result))
    report(result)
    return result

In [ ]:
ARCH = {"1x32": [32], "1x128": [128], "2x64": [64, 64], "2x128": [128, 128],
        "3x128": [128, 128, 128], "3x256": [256, 256, 256]}
EXPERIMENTS  = [(f"{a}_relu_d0.0", h, "relu", 0.0) for a, h in ARCH.items()]                     # sizes
EXPERIMENTS += [(f"{a}_{act}_d0.0", ARCH[a], act, 0.0) for a in ("1x32", "2x128") for act in ("sigmoid", "tanh")]  # activations
EXPERIMENTS += [(f"{a}_relu_d{p}", ARCH[a], "relu", p) for a in ("2x128", "3x256") for p in (0.2, 0.5)]            # dropout
print(len(EXPERIMENTS), "experiments")

for exp in EXPERIMENTS:          # slow part: safe to stop and restart, finished runs are skipped
    run_experiment(*exp)

14 experiments
1x32_relu_d0.0           epochs=40     84s  val_acc=0.7826  test_acc=0.7828
1x128_relu_d0.0          epochs=40     82s  val_acc=0.9978  test_acc=0.9980
2x64_relu_d0.0           epochs=30     75s  val_acc=0.9959  test_acc=0.9961
2x128_relu_d0.0          epochs=40    153s  val_acc=0.9999  test_acc=0.9999
3x128_relu_d0.0          epochs=40    121s  val_acc=0.9999  test_acc=0.9998
3x256_relu_d0.0          epochs=40    147s  val_acc=0.9999  test_acc=0.9999
1x32_sigmoid_d0.0        epochs=40     51s  val_acc=0.7252  test_acc=0.7264
1x32_tanh_d0.0           epochs=40     59s  val_acc=0.8926  test_acc=0.8938
2x128_sigmoid_d0.0       epochs=40    102s  val_acc=0.9999  test_acc=0.9999
2x128_tanh_d0.0          epochs=40     94s  val_acc=1.0000  test_acc=1.0000
2x128_relu_d0.2          epochs=40    109s  val_acc=0.9974  test_acc=0.9973
2x128_relu_d0.5          epochs=40    107s  val_acc=0.9402  test_acc=0.9399
3x256_relu_d0.2          epochs=21    231s  val_acc=0.9995  test_acc=0.99

## 3. Results

In [ ]:
def summary():
    rows = [json.loads(p.read_text()) for p in RESULTS.glob("*.json")]
    df = pd.DataFrame(rows).drop(columns=["history", "recall"])
    df["hidden"] = df["hidden"].astype(str)
    return df.sort_values(["val_acc", "val_loss"], ascending=[False, True]).reset_index(drop=True)

def plot_run(name):
    r = json.loads((RESULTS / f"{name}.json").read_text()); h = r["history"]; ep = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for a, key, title in ((ax[0], "accuracy", "accuracy"), (ax[1], "loss", "cross-entropy loss")):
        a.plot(ep, h[key], label="train"); a.plot(ep, h["val_" + key], label="validation")
        a.set_xlabel("epoch"); a.set_title(title); a.legend()
    fig.suptitle(f"{name}   (test accuracy {r['test_acc']:.4f})"); fig.tight_layout(); plt.show()

def per_class(name):
    r = json.loads((RESULTS / f"{name}.json").read_text())
    return pd.Series(r["recall"], index=CLASSES, name=name).round(3)

summary()

## 4. Cross-validation of the best network

In [ ]:
def cross_validate(hidden, activation="relu", dropout=0.0, k=5):
    """Balance only inside each fold's training part, so duplicated rare hands never leak into the fold being scored."""
    scores = []
    for fold, (tr, va) in enumerate(StratifiedKFold(k, shuffle=True, random_state=SEED).split(X_tr, y_tr), 1):
        Xb, yb = balance(X_tr[tr], y_tr[tr])
        keras.utils.set_random_seed(SEED + fold)
        model = build_model(hidden, activation, dropout)
        stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
        model.fit(Xb, to_categorical(yb, 10), validation_data=(X_tr[va], to_categorical(y_tr[va], 10)),
                  epochs=MAX_EPOCHS, batch_size=BATCH, callbacks=[stop], verbose=0)
        pred = model.predict(X_tr[va], batch_size=4096, verbose=0).argmax(1)
        scores.append(float((pred == y_tr[va]).mean()))
        print(f"fold {fold}: accuracy {scores[-1]:.4f}")
    print(f"mean {np.mean(scores):.4f} +/- {np.std(scores):.4f}")
    return scores

## 5. Export and reload of the best network

In [ ]:
best_name = summary().loc[0, "name"]                           # highest validation accuracy (ties: lowest validation loss)
keras.models.load_model(MODELS / f"{best_name}.keras").save("best_poker_model.keras")
print("exported:", best_name)

In [ ]:
# Load the exported model and evaluate it on a dataset read from file (raw hands, same columns as poker_hands.csv)
loaded = keras.models.load_model("best_poker_model.keras")
hands = pd.read_csv(TEST_FILE)
pred = loaded.predict(encode(hands), batch_size=4096, verbose=0).argmax(1)
print(f"accuracy of the loaded model on {len(hands):,} hands from {TEST_FILE}: {(pred == labels(hands)).mean():.4f}")